In [1]:
model_name = "vit-ragdoll"


import iree
import iree.compiler
import iree.runtime
import torch
import numpy as np
from torch import nn
from torchvision import models

import warnings
warnings.filterwarnings("ignore")

from timeit import timeit as ti
import seaborn as sns
import matplotlib.pyplot as plt
import pandas as pd

from ragdoll.compiler import *
from ragdoll.benchmark import *
import ragdoll

def get_dataframe(backward, item):
    return pd.concat([
        pd.DataFrame({
            "time": backward,
            "pass": "Backward",
            "item": item
        }, index=[0]),
    ])

def timeit(stmt, n=1):
    return ti(stmt, globals=globals(), number=n) * 1000 / n



BENCHMARK_REPEAT=33
df = pd.DataFrame()

def load_executable(fb_file):
    config = iree.runtime.system_api.Config("cuda")
    vmi = iree.runtime.VmInstance()
    # replace compile with args of fatbin type
    # fb_file = ragdoll.compile(mlir, "gpu", "input", "codegen", benchmark=True)
    with open(fb_file, 'rb') as f:
        binary_data = f.read()
    vmm = iree.runtime.VmModule.from_flatbuffer(vmi, binary_data)
    vmo = iree.runtime.load_vm_module(vmm, config)
    return vmo

import torch
from torch import nn
from torchvision import models
import pandas as pd

MAGIC_NUM = 7777e-5

device = torch.device("cuda:0")
model = models.vit_l_16().train(False)
model.load_state_dict({k: torch.ones_like(v) * MAGIC_NUM for k, v in model.state_dict().items()})
model = model.to(device)

df = pd.DataFrame()

!cpupower frequency-set --governor performance

Setting cpu: 0
Setting cpu: 1
Setting cpu: 2
Setting cpu: 3
Setting cpu: 4
Setting cpu: 5
Setting cpu: 6
Setting cpu: 7
Setting cpu: 8
Setting cpu: 9
Setting cpu: 10
Setting cpu: 11
Setting cpu: 12
Setting cpu: 13
Setting cpu: 14
Setting cpu: 15
Setting cpu: 16
Setting cpu: 17
Setting cpu: 18
Setting cpu: 19
Setting cpu: 20
Setting cpu: 21
Setting cpu: 22
Setting cpu: 23
Setting cpu: 24
Setting cpu: 25
Setting cpu: 26
Setting cpu: 27
Setting cpu: 28
Setting cpu: 29
Setting cpu: 30
Setting cpu: 31
Setting cpu: 32
Setting cpu: 33
Setting cpu: 34
Setting cpu: 35
Setting cpu: 36
Setting cpu: 37
Setting cpu: 38
Setting cpu: 39
Setting cpu: 40
Setting cpu: 41
Setting cpu: 42
Setting cpu: 43
Setting cpu: 44
Setting cpu: 45
Setting cpu: 46
Setting cpu: 47


In [2]:
for bs in range(1, 10):
    dyn_model = torch.compile(model, backend="inductor")
    image = torch.randn(bs, 3, 224, 224, requires_grad=True)
    image = image.to(device)
    output = dyn_model(image)
    grad = torch.randn_like(output)
    
    torch.cuda.reset_peak_memory_stats(device=device)
    before = torch.cuda.memory_allocated(device=device)
    print("measuring #", bs)
    baseline_f = timeit("dyn_model(image.to(device))", 33)
    _ = timeit("torch.autograd.grad(output.to(device), [image.to(device)], grad.to(device), retain_graph=True)", 1)
    baseline_b = timeit("torch.autograd.grad(output.to(device), [image.to(device)], grad.to(device), retain_graph=True)", 13)
    #print(baseline_f)
    print(baseline_b)

    # 记录操作后的峰值内存使用情况
    peak_memory = torch.cuda.max_memory_allocated(device=device)
    
    # 显示结果
    print(f"Memory used before operation: {before / (1024**2):.2f} MB")
    print(f"Peak memory usage: {peak_memory / (1024**2):.2f} MB")

measuring # 1
35.50636811325183
Memory used before operation: 1467.70 MB
Peak memory usage: 2638.92 MB
measuring # 2
57.06724409873669
Memory used before operation: 1829.41 MB
Peak memory usage: 3003.31 MB
measuring # 3
80.12509023627409
Memory used before operation: 2134.53 MB
Peak memory usage: 3312.31 MB
measuring # 4
102.81065665185452
Memory used before operation: 2466.19 MB
Peak memory usage: 3763.73 MB
measuring # 5
116.60572213049119
Memory used before operation: 2816.67 MB
Peak memory usage: 4464.91 MB
measuring # 6
147.38109550223902
Memory used before operation: 3078.29 MB
Peak memory usage: 4985.90 MB
measuring # 7
161.64509407602824
Memory used before operation: 3416.87 MB
Peak memory usage: 5661.51 MB
measuring # 8
184.40625093017633
Memory used before operation: 3709.94 MB
Peak memory usage: 6250.26 MB


OutOfMemoryError: CUDA out of memory. Tried to allocate 28.00 MiB (GPU 0; 7.78 GiB total capacity; 6.40 GiB already allocated; 9.94 MiB free; 7.55 GiB reserved in total by PyTorch) If reserved memory is >> allocated memory try setting max_split_size_mb to avoid fragmentation.  See documentation for Memory Management and PYTORCH_CUDA_ALLOC_CONF

Process ForkProcess-26:
Process ForkProcess-29:
Process ForkProcess-2:
Process ForkProcess-11:
Process ForkProcess-7:
Process ForkProcess-22:
Process ForkProcess-27:
Process ForkProcess-12:
Process ForkProcess-6:
Process ForkProcess-9:
Process ForkProcess-3:
Process ForkProcess-32:
Process ForkProcess-19:
Process ForkProcess-10:
Process ForkProcess-21:
Process ForkProcess-28:
Process ForkProcess-25:
Process ForkProcess-24:
Process ForkProcess-23:
Process ForkProcess-18:
Process ForkProcess-20:
Process ForkProcess-30:
Process ForkProcess-31:
Process ForkProcess-17:
Process ForkProcess-5:
Process ForkProcess-4:
Traceback (most recent call last):
Traceback (most recent call last):
Traceback (most recent call last):
Traceback (most recent call last):
Process ForkProcess-16:
Traceback (most recent call last):
Traceback (most recent call last):
Process ForkProcess-8:
Traceback (most recent call last):
Traceback (most recent call last):
Process ForkProcess-1:
Traceback (most recent call last)

In [2]:
from timeit import timeit as ti
def timeit(stmt, n=1):
    return ti(stmt, globals=globals(), number=n) * 1000 / n
for bs in range(1, 10):
    image = torch.randn(bs, 3, 224, 224, requires_grad=True)
    image = image.to(device)
    output = model(image)
    grad = torch.randn_like(output)
    torch.cuda.reset_peak_memory_stats(device=device)
    before = torch.cuda.memory_allocated(device=device)
    print("measuring #", bs)
    rep_times = 3
    baseline_f = timeit("model(image)", 30)
    baseline_b = timeit("torch.autograd.grad([output], [image], [grad], retain_graph=True)", rep_times)

    #print(baseline_f)
    print(baseline_b)
    
    # 记录操作后的峰值内存使用情况
    peak_memory = torch.cuda.max_memory_allocated(device=device)
    
    # 显示结果
    print(f"Memory used before operation: {before / (1024**2):.2f} MB")
    print(f"Peak memory usage: {peak_memory / (1024**2):.2f} MB")

measuring # 1
34.8927266895771
Memory used before operation: 1472.75 MB
Peak memory usage: 1776.55 MB
measuring # 2
48.46886917948723
Memory used before operation: 1813.91 MB
Peak memory usage: 2452.26 MB
measuring # 3
68.12773800144593
Memory used before operation: 2128.66 MB
Peak memory usage: 3081.03 MB
measuring # 4
88.25951690475146
Memory used before operation: 2446.13 MB
Peak memory usage: 3718.60 MB
measuring # 5
103.90832057843606
Memory used before operation: 2817.59 MB
Peak memory usage: 4461.97 MB
measuring # 6
127.21805895368259
Memory used before operation: 3080.31 MB
Peak memory usage: 4988.34 MB
measuring # 7
143.42199607441822
Memory used before operation: 3416.82 MB
Peak memory usage: 5660.72 MB
measuring # 8
159.896370023489
Memory used before operation: 3710.24 MB
Peak memory usage: 6249.86 MB
measuring # 9
176.90096547206244
Memory used before operation: 4045.04 MB
Peak memory usage: 6917.54 MB


In [9]:
import gc
model_file_base = "vit.mlir"
strategy = "heuristic"
ragdoll_results = []
ragdoll_throughput = []
for bs in range(1, 37):
    print("measuring #", bs)
    model_file = model_file_base + ".bs{}".format(bs)
    source_file = model_file + ".{}".format(strategy)
    target_file = source_file + ".vmfb"
    
    # gen model with specified batch-size
    !ragdoll-opt {model_file_base} --ragdoll-autodiff-prepare-batch-size=batchsize={bs} > {model_file}
    
    !ragdoll-opt {model_file}  \
    --canonicalize \
    --enable-cse-in-legalizer \
    --symbol-dce \
    --ragdoll-autodiff-vjp-public-functions='strategy=heuristic' \
    --ragdoll-autodiff-vjp \
    --inline \
    --ragdoll-autodiff-inline-function-call \
    --ragdoll-initialisation \
    --eliminate-empty-tensors \
    --ragdoll-legalise-to-iree-compatibility \
    --ragdoll-raise-linalg-to-tosa \
    --ragdoll-forward-func-removal \
    --canonicalize \
    --cse > {source_file}
    
    !iree-compile {source_file} \
    -o {target_file} \
    --iree-codegen-llvm-distribution-size=8 \
    --iree-hal-target-backends=cuda \
    --iree-hal-benchmark-dispatch-repeat-count={BENCHMARK_REPEAT} \
    --iree-hal-cuda-llvm-target-arch=sm_70
    
    
    
    #ragdoll_binary = load_executable(target_file)


    #try:
        #f1 = timeit("ragdoll_binary.forward(image_np_t)") / BENCHMARK_REPEAT
        #print('ragdoll-opt1-gpu-forward in timeit: ', f1)
    #"""
    !iree-benchmark-module \
    --module={target_file} \
    --device=cuda \
    --function=dforward \
    --input={bs}x1000xf32 \
    --batch_size={BENCHMARK_REPEAT} \
    --benchmark_repetitions=1 \
    --batch_concurrency=1 \
    --benchmark_min_time=0.1s \
    --print_statistics=true
    #"""
    b1 = ragdoll_model_benchmark(
        target_file,
        "dforward",
        [(bs, 1000)],
        device='gpu',
        warmups=11,
        repetitions=BENCHMARK_REPEAT, 
        measure_count=1,
        record_mem=True,
        verbose=False)
    b1 = np.mean(b1)
    print(b1)
    """
    b1 = timeit("ragdoll_binary.dforward(grad_np)") / BENCHMARK_REPEAT



    
    ragdoll_results.append(b1)
    ragdoll_throughput.append(bs/b1)
    """


measuring # 1
2024-03-30T10:30:26+08:00
Running /root/miniconda3/envs/albert-research-py310/lib/python3.10/site-packages/iree/_runtime_libs/iree-benchmark-module
Run on (48 X 4354.15 MHz CPU s)
CPU Caches:
  L1 Data 32 KiB (x24)
  L1 Instruction 32 KiB (x24)
  L2 Unified 512 KiB (x24)
  L3 Unified 16384 KiB (x8)
Load Average: 0.37, 6.12, 7.92
---------------------------------------------------------------------------------------------
Benchmark                                   Time             CPU   Iterations UserCounters...
---------------------------------------------------------------------------------------------
BM_dforward/process_time/real_time       29.0 ms         29.0 ms           33 items_per_second=34.433/s
[[ iree_hal_allocator_t memory statistics ]]
  HOST_LOCAL:            0B peak /            0B allocated /            0B freed /            0B live
DEVICE_LOCAL:     48480416B peak /     48476416B allocated /     48476416B freed /            0B live
[BenchmarkResult(ben

In [1]:
!sudo cpupower frequency-set --governor powersave

Setting cpu: 0
Setting cpu: 1
Setting cpu: 2
Setting cpu: 3
Setting cpu: 4
Setting cpu: 5
Setting cpu: 6
Setting cpu: 7
Setting cpu: 8
Setting cpu: 9
Setting cpu: 10
Setting cpu: 11
Setting cpu: 12
Setting cpu: 13
Setting cpu: 14
Setting cpu: 15
Setting cpu: 16
Setting cpu: 17
Setting cpu: 18
Setting cpu: 19
Setting cpu: 20
Setting cpu: 21
Setting cpu: 22
Setting cpu: 23
Setting cpu: 24
Setting cpu: 25
Setting cpu: 26
Setting cpu: 27
Setting cpu: 28
Setting cpu: 29
Setting cpu: 30
Setting cpu: 31
Setting cpu: 32
Setting cpu: 33
Setting cpu: 34
Setting cpu: 35
Setting cpu: 36
Setting cpu: 37
Setting cpu: 38
Setting cpu: 39
Setting cpu: 40
Setting cpu: 41
Setting cpu: 42
Setting cpu: 43
Setting cpu: 44
Setting cpu: 45
Setting cpu: 46
Setting cpu: 47
